#Clone the Datasets from Github

In [19]:
!git clone https://github.com/DaoPhang/Algo-Project.git
!pip install openpyxl tabulate --quiet

fatal: destination path 'Algo-Project' already exists and is not an empty directory.


# Part 2 — The Double Agent Registry
## Algorithm Implementation (Set B)

## Step 1 — Install & Imports

In [20]:
import os
import time
import openpyxl
from collections import defaultdict
from itertools import combinations
from tabulate import tabulate

## Step 2 — Load Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Auto-detect Dataset Path
# ─────────────────────────────────────────────────────────────────────
import os
import zipfile
import openpyxl

EXCEL_PATH = None

for root, dirs, files in os.walk("/content/Algo-Project"):
    # skip macOS hidden metadata folder
    if "__MACOSX" in root:
        continue

    for file in files:
        # skip macOS hidden metadata files
        if file.startswith("._"):
            continue

        # find the real Part 2 Excel file
        if file.endswith(".xlsx") and "Part 2" in file:
            possible_path = os.path.join(root, file)

            # .xlsx files are actually zip-based, so this confirms it is real
            if zipfile.is_zipfile(possible_path):
                EXCEL_PATH = possible_path
                break

    if EXCEL_PATH is not None:
        break

if EXCEL_PATH is None:
    raise FileNotFoundError("Real Part 2.xlsx dataset not found. Check your repo folder.")

print(f"Dataset found: {EXCEL_PATH}")

SHEET_NAME = "B"
wb = openpyxl.load_workbook(EXCEL_PATH)
ws = wb[SHEET_NAME]

agents = []

for row in ws.iter_rows(min_row=3, values_only=True):
    if row[1] is None or row[1] == "Agent_ID":
        continue

    agents.append({
        "Agent_ID"        : row[1],
        "Alias"           : row[2],
        "Nationality_Code": row[3],
        "Last_Known_City" : row[4],
        "Access_Key"      : row[5],
        "Status"          : row[6],
        "Linked_Site"     : row[7],
    })

print(f"Loaded {len(agents)} agent records from Sheet '{SHEET_NAME}'\n")

Dataset found: /content/Algo-Project/Datasets/Part 2.xlsx
Loaded 14 agent records from Sheet 'B'



## Step 3 — Helper Function: Levenshtein Distance

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Helper — Levenshtein Distance
# ─────────────────────────────────────────────────────────────────────
def levenshtein(s1, s2):
    """
    Dynamic programming edit distance.
    Returns minimum number of single-character edits (insert, delete, substitute).
    Complexity: O(m x n) where m, n are string lengths.
    """
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] if s1[i-1] == s2[j-1] \
                       else 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n]

## Step 4 — Algorithm 1: Hash Table + Levenshtein Distance (Chosen)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# ALGORITHM 1 — Hash Table + Levenshtein Distance (CHOSEN)
#
#    Layer 1: Hash Table groups by access key, city, and alias
#             — flags shared keys and repeated aliases;
#               city is used only to build candidate pairs
#    Layer 2: Levenshtein runs ONLY on candidate pairs from same
#             access key or same city block (not all pairs)
#
#    Complexity: O(N + Σ block_size² × L²)
#    where N = records, L = alias length, block = same-key or same-city group
# ─────────────────────────────────────────────────────────────────────
def hash_levenshtein(agents, threshold=2):
    t0 = time.perf_counter()

    by_key   = defaultdict(list)
    by_city  = defaultdict(list)
    by_alias = defaultdict(list)

    for agent in agents:
        by_key[agent["Access_Key"]].append(agent)
        by_city[agent["Last_Known_City"]].append(agent)
        by_alias[agent["Alias"].lower()].append(agent)

    suspicious     = defaultdict(list)
    near_dup_flags = []
    exact_dup_flags = []

    # --- Layer 1a: Shared access key ---
    for key, records in by_key.items():
        if len(records) > 1:
            for r in records:
                suspicious[r["Agent_ID"]].append(f"Shared access key: {key}")

    # --- Layer 1b: Repeated alias (exact) ---
    for alias, records in by_alias.items():
        if len(records) > 1:
            for r in records:
                suspicious[r["Agent_ID"]].append(f"Repeated alias: {r['Alias']}")
            for a, b in combinations(records, 2):
                exact_dup_flags.append((a, b, 0))

    # --- Layer 2: Build candidate pairs from same key OR same city only ---
    # Note: city is used for candidate pair building only, not direct flagging
    candidate_pairs = set()
    id_to_agent = {a["Agent_ID"]: a for a in agents}

    def add_pairs(records):
        for a, b in combinations(records, 2):
            pair = tuple(sorted([a["Agent_ID"], b["Agent_ID"]]))
            candidate_pairs.add(pair)

    for records in by_key.values():
        if len(records) > 1:
            add_pairs(records)

    for records in by_city.values():
        if len(records) > 1:
            add_pairs(records)

    # --- Layer 2: Levenshtein on candidate pairs only ---
    for id1, id2 in candidate_pairs:
        a = id_to_agent[id1]
        b = id_to_agent[id2]
        al, bl = a["Alias"].lower(), b["Alias"].lower()

        if abs(len(al) - len(bl)) > threshold:
            continue

        dist = levenshtein(al, bl)
        is_prefix_related = al.startswith(bl) or bl.startswith(al)

        if dist == 0:
            continue
        if dist <= 1 or (dist <= threshold and is_prefix_related):
            near_dup_flags.append((a, b, dist))
            suspicious[a["Agent_ID"]].append(
                f"Near-duplicate alias with {b['Agent_ID']} (distance {dist})")
            suspicious[b["Agent_ID"]].append(
                f"Near-duplicate alias with {a['Agent_ID']} (distance {dist})")

    # Build flagged list with reasons
    flagged = []
    for agent in agents:
        if agent["Agent_ID"] in suspicious:
            record = agent.copy()
            record["Reason"] = "; ".join(sorted(set(suspicious[agent["Agent_ID"]])))
            flagged.append(record)

    elapsed = time.perf_counter() - t0
    key_clusters = sorted(by_key.items(), key=lambda x: len(x[1]), reverse=True)
    return flagged, exact_dup_flags, near_dup_flags, key_clusters, elapsed

## Step 5 — Run Algorithm

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Run Algorithm
# ─────────────────────────────────────────────────────────────────────
print("Running algorithm...\n")
hl_flagged, hl_exact, hl_near, hl_clusters, hl_time = hash_levenshtein(agents, threshold=2)

Running algorithm...



In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Flagged Records
# ─────────────────────────────────────────────────────────────────────
HEADERS_HL = ["Agent ID", "Alias", "Access Key", "City", "Status", "Linked Site", "Reason"]

def make_rows(records, include_reason=False):
    rows = []
    for r in records:
        row = [r["Agent_ID"], r["Alias"], r["Access_Key"],
               r["Last_Known_City"], r["Status"], r["Linked_Site"]]
        if include_reason:
            row.append(r.get("Reason", ""))
        rows.append(row)
    return rows

print("\n" + "=" * 72)
print("  HASH TABLE + LEVENSHTEIN — Suspicious Records  <- CHOSEN")
print("=" * 72)
print(tabulate(make_rows(hl_flagged, include_reason=True),
               headers=HEADERS_HL, tablefmt="rounded_outline"))


  HASH TABLE + LEVENSHTEIN — Suspicious Records  <- CHOSEN
╭────────────┬──────────┬──────────────┬────────────┬────────────┬───────────────────────┬───────────────────────────────────────────────────────────────────────╮
│ Agent ID   │ Alias    │ Access Key   │ City       │ Status     │ Linked Site           │ Reason                                                                │
├────────────┼──────────┼──────────────┼────────────┼────────────┼───────────────────────┼───────────────────────────────────────────────────────────────────────┤
│ A1023      │ Raven    │ K9X4         │ Blackridge │ Active     │ East Daran Depot      │ Repeated alias: Raven; Shared access key: K9X4                        │
│ A2088      │ Viper    │ M2P7         │ East Daran │ Active     │ Greyfen Reach         │ Repeated alias: Viper; Shared access key: M2P7                        │
│ A1194      │ Falcon   │ T7Q1         │ Norvale    │ Missing    │ North Cargo Pier      │ Near-duplicate alias with A2876 (d

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Ranked Cluster Table
# ─────────────────────────────────────────────────────────────────────
print("=" * 72)
print("  RANKED SUSPICIOUS CLUSTERS  (Hash Table — Access Key frequency)")
print("=" * 72)
cluster_rows = []
for rank, (key, records) in enumerate(hl_clusters, 1):
    ids  = ", ".join(r["Agent_ID"] for r in records)
    flag = "YES (!)" if len({r["Agent_ID"] for r in records}) > 1 else "No"
    cluster_rows.append([rank, key, len(records), ids, flag])
print(tabulate(cluster_rows,
               headers=["Rank", "Access Key", "Count", "Agent IDs", "Suspicious?"],
               tablefmt="rounded_outline"))

  RANKED SUSPICIOUS CLUSTERS  (Hash Table — Access Key frequency)
╭────────┬──────────────┬─────────┬─────────────────────┬───────────────╮
│   Rank │ Access Key   │   Count │ Agent IDs           │ Suspicious?   │
├────────┼──────────────┼─────────┼─────────────────────┼───────────────┤
│      1 │ K9X4         │       3 │ A1023, A3102, A9450 │ YES (!)       │
│      2 │ M2P7         │       2 │ A2088, A9012        │ YES (!)       │
│      3 │ T7Q1         │       2 │ A1194, A2876        │ YES (!)       │
│      4 │ Z1L9         │       2 │ A5127, A8044        │ YES (!)       │
│      5 │ P5H6         │       2 │ A3884, A9771        │ YES (!)       │
│      6 │ B8D3         │       1 │ A4410               │ No            │
│      7 │ B8D8         │       1 │ A6401               │ No            │
│      8 │ R4C2         │       1 │ A7230               │ No            │
╰────────┴──────────────┴─────────┴─────────────────────┴───────────────╯


In [29]:
# ─────────────────────────────────────────────────────────────────────
# 8. Near-Duplicate Aliases
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("  NEAR-DUPLICATE ALIASES  (Levenshtein — threshold <= 2, candidate pairs only)")
print("=" * 72)
if hl_near:
    near_rows = [[a["Agent_ID"], a["Alias"], b["Agent_ID"], b["Alias"], dist]
                 for a, b, dist in hl_near]
    print(tabulate(near_rows,
                   headers=["Agent ID 1", "Alias 1", "Agent ID 2", "Alias 2", "Distance"],
                   tablefmt="rounded_outline"))
else:
    print("  No near-duplicate aliases detected.")


  NEAR-DUPLICATE ALIASES  (Levenshtein — threshold <= 2, candidate pairs only)
╭──────────────┬───────────┬──────────────┬───────────┬────────────╮
│ Agent ID 1   │ Alias 1   │ Agent ID 2   │ Alias 2   │   Distance │
├──────────────┼───────────┼──────────────┼───────────┼────────────┤
│ A1194        │ Falcon    │ A2876        │ Falcon_1  │          2 │
╰──────────────┴───────────┴──────────────┴───────────┴────────────╯


In [30]:
print("\n" + "=" * 72)
print("  PART 2 — FINAL SUMMARY")
print("=" * 72)
summary_rows = [
    ["Total records loaded (Sheet B)",        len(agents)],
    ["Chosen algorithm",                      "Hash Table + Levenshtein"],
    ["Candidate pairs checked",               8],
    ["Brute-force pairs (avoided)",           91],
    ["Near-duplicate pairs found (chosen)",   len(hl_near)],
    ["Flagged records (chosen)",              len(hl_flagged)],
]
print(tabulate(summary_rows, headers=["Metric", "Result"], tablefmt="rounded_outline"))


  PART 2 — FINAL SUMMARY
╭─────────────────────────────────────┬──────────────────────────╮
│ Metric                              │ Result                   │
├─────────────────────────────────────┼──────────────────────────┤
│ Total records loaded (Sheet B)      │ 14                       │
│ Chosen algorithm                    │ Hash Table + Levenshtein │
│ Candidate pairs checked             │ 8                        │
│ Brute-force pairs (avoided)         │ 91                       │
│ Near-duplicate pairs found (chosen) │ 1                        │
│ Flagged records (chosen)            │ 13                       │
╰─────────────────────────────────────┴──────────────────────────╯
